### TPA program on MeerKat 

In this notebook, we analyze the pulsars observed by the TPA program on Meerkat as detailed in the paper by [Posselt et al. (2023)](https://academic.oup.com/mnras/article/520/3/4582/7049638). We will use the fluxes from these observations to compare the observed and simulated populations in order to constrain the intrinsic luminosity law of neutron stars.

In [ ]:
import argparse
import logging
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utilities.plot_settings

In [ ]:
# Read the full ATNF catalogue. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    '../../data/observations/atnf_full_nobinary_06-08-2024.csv',
    delimiter=";",
    header=[0, 1],
)

In [ ]:
# Read in the pulsar data from the TPA program.
df_meerkat = pd.read_csv(
    '../../data/observations/meerkat_tpa_posselt_2023.csv',
    delimiter=",",
)

In [ ]:
df_atnf.columns = df_atnf.columns.get_level_values(0)
# Select only stars with measured P, Pdot, DM and radio flux.
# We also select only those that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["P0"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["P1"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["ASSOC"].str.match("|".join(discard))]

# Select only isolated, non-recycled neutron stars, i.e., those with Pdot > 1e-19.
df_atnf = df_atnf[df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19]

In [ ]:
# Parkes multibeam pulsar survey database.
df_atnf_pmps = df_atnf[df_atnf["SURVEY"].str.contains("pksmb")]

# Extracting Galactic longitude, latitude, period and period derivative.
RA_pmps_obs = df_atnf_pmps["RAJD"].to_numpy().astype(np.float64)
DEC_pmps_obs = df_atnf_pmps["DECJD"].to_numpy().astype(np.float64)
l_pmps_obs = df_atnf_pmps["Gl"].to_numpy().astype(np.float64)
b_pmps_obs = df_atnf_pmps["Gb"].to_numpy().astype(np.float64)
P_pmps_obs = df_atnf_pmps["P0"].to_numpy().astype(np.float64)
Pdot_pmps_obs = df_atnf_pmps["P1"].to_numpy().astype(np.float64)

# Converting galactic latitude into the range [-180., 180].
l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] = (
    l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] - 360.0
)

# Select only pulsars falling into the Parkes multibeam sky coverage where completeness is above 90%.
# See Lorimer et al. (2006) for details.
cond = (
    (l_pmps_obs > -100.0)
    & (l_pmps_obs < 50.0)
    & (np.abs(b_pmps_obs) < 5.0)
)

In [ ]:
# Merge the Meerkat TPA program data with the PMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurement for the PMPS pulsars.
df_meerkat_pmps = pd.merge(
    df_meerkat, df_atnf_pmps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_pmps = df_meerkat_pmps.dropna(subset=["ch6flux"])

RA_pmps_obs = RA_pmps_obs[cond]
DEC_pmps_obs = DEC_pmps_obs[cond]
P_pmps_obs = P_pmps_obs[cond]
Pdot_pmps_obs = Pdot_pmps_obs[cond]

# We convert the [Jy] to [mJy] to compare with simulations.
S1400_pmps_meerkat = (
    df_meerkat_pmps["ch6flux"].to_numpy().astype(np.float64) / 1000
)
P_pmps_meerkat = df_meerkat_pmps["P0"].to_numpy().astype(np.float64)
Pdot_pmps_meerkat = df_meerkat_pmps["P1"].to_numpy().astype(np.float64)

In [ ]:
# Swinburne multibeam pulsar survey database.
df_atnf_smps = df_atnf[df_atnf["SURVEY"].str.contains("pkssw")]

# Extracting Galactic longitude, latitude, right ascension and declination, period and period derivative.
RA_smps_obs = df_atnf_smps["RAJD"].to_numpy().astype(np.float64)
DEC_smps_obs = df_atnf_smps["DECJD"].to_numpy().astype(np.float64)
l_smps_obs = df_atnf_smps["Gl"].to_numpy().astype(np.float64)
P_smps_obs = df_atnf_smps["P0"].to_numpy().astype(np.float64)
Pdot_smps_obs = df_atnf_smps["P1"].to_numpy().astype(np.float64)

# Converting galactic latitude into the range [-180., 180].
l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] = (
    l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] - 360.0
)

# Select only pulsars falling into the Swinburne sky coverage where completeness is above 90%.
# See Edwards et al. (2001) and Jacoby et al. (2009) for details.
cond = (l_smps_obs > -100.0) & (l_smps_obs < 50.0)

# Merge the Meerkat TPA program data with the SMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements for the SMPS pulsars.
df_meerkat_smps = pd.merge(
    df_meerkat, df_atnf_smps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_smps = df_meerkat_smps.dropna(subset=["ch6flux"])

RA_smps_obs = RA_smps_obs[cond]
DEC_smps_obs = DEC_smps_obs[cond]
P_smps_obs = P_smps_obs[cond]
Pdot_smps_obs = Pdot_smps_obs[cond]

# We convert the [Jy] to [mJy] to compare with simulations.
S1400_smps_meerkat = (
    df_meerkat_smps["ch6flux"].to_numpy().astype(np.float64) / 1000
)
P_smps_meerkat = df_meerkat_smps["P0"].to_numpy().astype(np.float64)
Pdot_smps_meerkat = df_meerkat_smps["P1"].to_numpy().astype(np.float64)

In [ ]:
# HTRU pulsar survey database. Note that those HTRU pulsars in the ATNF Catalogue with P and Pdot
# measurements are from the low- and mid- latitude surveys only.
df_atnf_htru = df_atnf[df_atnf["SURVEY"].str.contains("htru_pks")]

# Merge the Meerkat TPA program data with the HTRU ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements for the HTRU pulsars.
df_meerkat_htru = pd.merge(
    df_meerkat, df_atnf_htru, left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_htru = df_meerkat_htru.dropna(subset=["ch6flux"])

RA_htru_obs = df_atnf_htru["RAJD"].to_numpy().astype(np.float64)
DEC_htru_obs = df_atnf_htru["DECJD"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"].to_numpy().astype(np.float64)

# We convert the [Jy] to [mJy] to compare with simulations.
S1400_htru_meerkat = (
    df_meerkat_htru["ch6flux"].to_numpy().astype(np.float64) / 1000
)
P_htru_meerkat = df_meerkat_htru["P0"].to_numpy().astype(np.float64)
Pdot_htru_meerkat = df_meerkat_htru["P1"].to_numpy().astype(np.float64)

In [ ]:
print(f"Number of overlapping pulsars in the TPA Meerkat program and PMPS: {len(df_meerkat_pmps)} ")

In [ ]:
print(f"Number of overlapping pulsars in the TPA Meerkat program and SMPS: {len(df_meerkat_smps)} ")

In [ ]:
print(f"Number of overlapping pulsars in the TPA Meerkat program and HTRU: {len(df_meerkat_htru)} ")

Due to the discrepancy between the number of pulsars in the TPA Meerkat program and the ATNF catalog, in the following we plot the distributions for Dispersion Measure (DM), Period, Period Derivative, and positions. This analysis is intended to identify any potential biases in the Meerkat program and to understand whther a direct comparisons between the two surveys is feasible.

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_pmps["RAJD"],
    df_meerkat_pmps["DECJD"],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS-MeerKAT",
)
ax.plot(
    RA_pmps_obs,
    DEC_pmps_obs,
    linestyle="None",
    marker="*",
    color="tab:blue",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS-ATNF",
)


ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(100, 300)
ax.set_ylim(-90.0, 17)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_smps["RAJD"],
    df_meerkat_smps["DECJD"],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS-MeerKAT",
)
ax.plot(
    RA_smps_obs,
    DEC_smps_obs,
    linestyle="None",
    marker="*",
    color="tab:blue",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS-ATNF",
)


ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

plt.show()
fig, ax = plt.subplots(figsize=(15, 8))


ax.plot(
    df_meerkat_htru["RAJD"],
    df_meerkat_htru["DECJD"],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU-MeerKAT",
)
ax.plot(
    RA_htru_obs,
    DEC_htru_obs,
    linestyle="None",
    marker="*",
    color="tab:blue",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU-ATNF",
)


ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(100, 300)
ax.set_ylim(-90.0, 0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
dm_edges = np.linspace(0, 2500, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_meerkat_pmps['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS-MeerKAT",
    rasterized=True,
)
ax.hist(
    df_atnf_pmps['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS-ATNF",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_meerkat_smps['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS-MeerKAT",
    rasterized=True,
)
ax.hist(
    df_atnf_smps['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS-ATNF",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_meerkat_htru['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU-MeerKAT",
    rasterized=True,
)
ax.hist(
    df_atnf_htru['DM'].to_numpy().astype(np.float64),
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU-ATNF",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
P_bins = np.logspace(-2.0, 2.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_pmps['P0'].to_numpy().astype(np.float64),
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS-ATNF",
)
ax.hist(
    df_meerkat_pmps['P0'].to_numpy().astype(np.float64),
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Observed PMPS-MeerKAT",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_smps['P0'].to_numpy().astype(np.float64),    
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS-ATNF",
)
ax.hist(
    df_meerkat_smps['P0'].to_numpy().astype(np.float64),    
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Observed SMPS-MeerKAT",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_htru['P0'].to_numpy().astype(np.float64),    
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS-ATNF",
)
ax.hist(
    df_meerkat_htru['P0'].to_numpy().astype(np.float64),    
    bins=P_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Observed HTRU-MeerKAT",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
Pdot_bins = np.logspace(-20.0, -8.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_pmps['P1'].to_numpy().astype(np.float64),
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS-ATNF",
)
ax.hist(
    df_meerkat_pmps['P1'].to_numpy().astype(np.float64),
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Observed PMPS-MeerKAT",
)

plt.xlabel(r"$\dot{P}$ [s/s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_smps['P1'].to_numpy().astype(np.float64),    
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS-ATNF",
)
ax.hist(
    df_meerkat_smps['P1'].to_numpy().astype(np.float64),    
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Observed SMPS-MeerKAT",
)

plt.xlabel(r"$\dot{P}$ [s/s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    df_atnf_htru['P1'].to_numpy().astype(np.float64),    
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS-ATNF",
)
ax.hist(
    df_meerkat_htru['P1'].to_numpy().astype(np.float64),    
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Observed HTRU-MeerKAT",
)

plt.xlabel(r"$\dot{P}$ [s/s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_pmps['S1400'].to_numpy().astype(np.float64),
    df_meerkat_pmps['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-PMPS flux")
plt.ylabel(r"MeerKAT-PMPS flux")
plt.grid()
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_smps['S1400'].to_numpy().astype(np.float64),
    df_meerkat_smps['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-SMPS flux")
plt.ylabel(r"MeerKAT-SMPS flux")
plt.grid()
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_htru['S1400'].to_numpy().astype(np.float64),
    df_meerkat_htru['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-HTRU flux")
plt.ylabel(r"MeerKAT-HTRU flux")
plt.grid()
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

As we observed that the TPA MeerKAT program does not detect all the pulsars observed by the PMPS, SMPS, and HTRU surveys, we wanted to check for any potential TPA biases. By examining the plots, it is clear that the biases in the TPA MeerKAT program are not different from those in the PMPS, SMPS, or HTRU surveys. This verification is crucial because we intend to use our simulated population, which accounts for the observational biases and systematics of these three surveys, along with the flux measurements from the TPA MeerKAT program, to constrain the intrinsic luminosity law of the neutron star population. Therefore, we need to make sure all surveys are consistent.